In [6]:
!pip install google-generativeai shap xgboost --quiet

In [21]:
import pickle
import json
import numpy as np
import pandas as pd
import shap
import google.generativeai as genai
from google.colab import files, userdata
from google.api_core import retry, exceptions # Added retry and exceptions

print("✓ All imports done")

✓ All imports done


In [22]:
import pickle
import json

# Load model
with open('/content/final_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Load SHAP explainer
with open('/content/shap_explainer.pkl', 'rb') as f:
    explainer = pickle.load(f)

# Load feature names
with open('/content/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

print("✓ Model loaded")
print("✓ Explainer loaded")
print(f"✓ Features: {feature_cols}")

✓ Model loaded
✓ Explainer loaded
✓ Features: ['util_rate', 'age', 'late_30_59', 'debt_ratio', 'monthly_income', 'open_credits', 'late_90', 'real_estate_loans', 'late_60_89', 'dependents', 'debt_to_income', 'total_late', 'has_late_history', 'income_per_dependent', 'high_util']


In [23]:
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini = genai.GenerativeModel("gemini-3-flash-preview")

print("✓ Gemini client ready")

✓ Gemini client ready


In [24]:

def shap_to_factors(shap_vals, feature_names, top_n=5):
    pairs = sorted(
        zip(shap_vals, feature_names),
        key=lambda x: abs(x[0]),
        reverse=True
    )[:top_n]

    labels = {
        'util_rate'            : ('معدل استخدام الائتمان',  'credit utilization rate'),
        'total_late'           : ('عدد مرات التأخر',         'late payment count'),
        'has_late_history'     : ('وجود تاريخ تأخر',         'late payment history'),
        'age'                  : ('العمر',                   'age'),
        'debt_ratio'           : ('نسبة الديون',             'debt ratio'),
        'monthly_income'       : ('الدخل الشهري',            'monthly income'),
        'debt_to_income'       : ('نسبة الدين للدخل',        'debt-to-income ratio'),
        'open_credits'         : ('عدد خطوط الائتمان',       'open credit lines'),
        'late_90'              : ('تأخر أكثر من 90 يوم',     '90+ days late'),
        'late_30_59'           : ('تأخر 30-59 يوم',          '30-59 days late'),
        'late_60_89'           : ('تأخر 60-89 يوم',          '60-89 days late'),
        'income_per_dependent' : ('الدخل لكل معال',          'income per dependent'),
        'high_util'            : ('استخدام ائتمان مرتفع',    'high credit usage'),
        'real_estate_loans'    : ('قروض عقارية',             'real estate loans'),
        'dependents'           : ('عدد المعالين',            'number of dependents'),
    }

    factors = []
    for shap_val, feat_name in pairs:
        ar_label, en_label = labels.get(feat_name, (feat_name, feat_name))
        factors.append({
            'feature'   : feat_name,
            'ar_label'  : ar_label,
            'en_label'  : en_label,
            'direction' : 'increases' if shap_val > 0 else 'decreases',
            'impact'    : round(abs(shap_val), 4),
            'shap_val'  : round(shap_val, 4)
        })
    return factors

print(" shap_to_factors() ready")

 shap_to_factors() ready


In [25]:

# Prompt Template
def build_prompt(client_data: dict, risk_score: float,
                 factors: list, language: str = 'en') -> str:

    risk_label    = (
        'High Risk'   if risk_score > 0.6 else
        'Medium Risk' if risk_score > 0.3 else
        'Low Risk'
    )
    risk_label_ar = (
        'مخاطرة عالية'   if risk_score > 0.6 else
        'مخاطرة متوسطة'  if risk_score > 0.3 else
        'مخاطرة منخفضة'
    )

    factors_text_en = "\n".join([
        f"- {f['en_label']}: {f['direction']} default risk (impact: {f['impact']})"
        for f in factors
    ])
    factors_text_ar = "\n".join([
        f"- {f['ar_label']}: {'يزيد' if f['direction'] == 'increases' else 'يقلل'} من مخاطر التعثر (تأثير: {f['impact']})"
        for f in factors
    ])

    client_summary_en = (
        f"Age: {client_data.get('age')}, "
        f"Monthly Income: ${client_data.get('monthly_income'):,.0f}, "
        f"Debt Ratio: {client_data.get('debt_ratio'):.2f}, "
        f"Credit Utilization: {client_data.get('util_rate'):.0%}, "
        f"Late Payments: {int(client_data.get('total_late', 0))} times"
    )
    client_summary_ar = (
        f"العمر: {client_data.get('age')} سنة، "
        f"الدخل الشهري: {client_data.get('monthly_income'):,.0f} دولار، "
        f"نسبة الديون: {client_data.get('debt_ratio'):.2f}، "
        f"معدل استخدام الائتمان: {client_data.get('util_rate'):.0%}، "
        f"مرات التأخر: {int(client_data.get('total_late', 0))} مرة"
    )

    if language == 'ar':
        return f"""أنت محلل مخاطر ائتمانية محترف في بنك. مهمتك كتابة تقرير واضح وموجز.

بيانات العميل:
{client_summary_ar}

نتيجة نموذج الذكاء الاصطناعي:
- درجة مخاطر التعثر: {risk_score:.1%}
- التصنيف: {risk_label_ar}

أهم العوامل المؤثرة:
{factors_text_ar}

اكتب تقريراً من 3 فقرات:
1. ملخص وضع العميل والتصنيف النهائي
2. أهم العوامل التي أثّرت في القرار وسببها
3. توصية واضحة للبنك (موافقة / رفض / موافقة بشروط)

اكتب بالعربية الفصحى بأسلوب مهني ومفهوم."""

    else:
        return f"""You are a professional credit risk analyst at a bank.

Client Profile:
{client_summary_en}

AI Model Output:
- Default Probability: {risk_score:.1%}
- Risk Classification: {risk_label}

Top Contributing Factors (SHAP):
{factors_text_en}

Write a 3-paragraph report:
1. Executive summary and final classification
2. Key drivers and why they matter
3. Clear recommendation (Approve / Reject / Approve with conditions)

Be professional, data-driven, and clear for non-technical stakeholders."""

print("✓ build_prompt() ready")

✓ build_prompt() ready


In [26]:
def explain_client(client_dict: dict, language: str = 'en') -> dict:

    # Input
    client_df  = pd.DataFrame([client_dict])[feature_cols]

    # Predict
    risk_score = float(model.predict_proba(client_df)[:, 1][0])

    # SHAP
    shap_exp  = explainer(client_df)
    shap_vals = shap_exp.values[0]
    factors   = shap_to_factors(shap_vals, feature_cols, top_n=5)

    # Prompt
    prompt = build_prompt(client_dict, risk_score, factors, language)

    # Gemini with retry mechanism
    @retry.Retry(predicate=retry.if_exception_type(exceptions.TooManyRequests),
                 initial=1.0,
                 multiplier=2.0,
                 maximum=60.0,
                 deadline=300.0) # Retry with exponential backoff
    def generate_content_with_retry(prompt_text):
        return gemini.generate_content(prompt_text)

    response   = generate_content_with_retry(prompt)
    llm_report = response.text

    risk_label = (
        'High Risk'   if risk_score > 0.6 else
        'Medium Risk' if risk_score > 0.3 else
        'Low Risk'
    )

    return {
        'risk_score' : round(risk_score, 4),
        'risk_label' : risk_label,
        'factors'    : factors,
        'llm_report' : llm_report,
        'language'   : language
    }

print("✓ explain_client() ready")

✓ explain_client() ready


In [27]:

#  Test: High Risk Client
high_risk_client = {
    'util_rate'            : 0.95,
    'age'                  : 28,
    'late_30_59'           : 3,
    'debt_ratio'           : 0.85,
    'monthly_income'       : 2500,
    'open_credits'         : 8,
    'late_90'              : 2,
    'real_estate_loans'    : 0,
    'late_60_89'           : 1,
    'dependents'           : 2,
    'debt_to_income'       : 0.85 / (2500 + 1),
    'total_late'           : 6,
    'has_late_history'     : 1,
    'income_per_dependent' : 2500 / 2,
    'high_util'            : 1
}

result_en = explain_client(high_risk_client, language='en')

print(f"Risk Score : {result_en['risk_score']:.1%}")
print(f"Risk Label : {result_en['risk_label']}")
print("\nTop Factors:")
for f in result_en['factors']:
    arrow = "↑" if f['direction'] == 'increases' else "↓"
    print(f"  {arrow} {f['en_label']} (impact: {f['impact']})")
print(f"\n{'='*50}")
print("LLM REPORT (English):")
print('='*50)
print(result_en['llm_report'])

Risk Score : 86.7%
Risk Label : High Risk

Top Factors:
  ↑ late payment count (impact: 2.6087000370025635)
  ↑ credit utilization rate (impact: 1.919800043106079)
  ↓ high credit usage (impact: 1.2829999923706055)
  ↑ debt ratio (impact: 0.7767999768257141)
  ↑ real estate loans (impact: 0.5419999957084656)

LLM REPORT (English):
**Credit Risk Assessment Report**

**Executive Summary and Risk Classification**
Following a comprehensive evaluation of the applicant’s profile, the credit risk is classified as **High Risk**. Our predictive modeling indicates a **Default Probability of 86.7%**, which significantly exceeds the bank’s acceptable risk threshold. The applicant’s financial profile—characterized by a low monthly income of $2,500 and a heavily leveraged balance sheet—suggests a severe inability to absorb new debt. The model's high confidence level reflects a profile that is currently in a state of high financial distress.

**Analysis of Key Risk Drivers**
The primary driver of thi

In [28]:

result_ar = explain_client(high_risk_client, language='ar')

print(f"درجة المخاطرة : {result_ar['risk_score']:.1%}")
print(f"التصنيف       : {result_ar['risk_label']}")
print(f"\n{'='*50}")
print("التقرير (عربي):")
print('='*50)
print(result_ar['llm_report'])

درجة المخاطرة : 86.7%
التصنيف       : High Risk

التقرير (عربي):
إليك تقرير تحليل المخاطر الائتمانية الخاص بالعميل:

**1. ملخص الوضع الائتماني والتصنيف النهائي:**
بناءً على التحليل الرقمي لبيانات العميل البالغ من العمر 28 عاماً، وبدخل شهري قدره 2,500 دولار، تم تصنيف الحالة الائتمانية ضمن فئة **"المخاطر العالية"**. أظهر نموذج الذكاء الاصطناعي احتمالية تعثر مرتفعة جداً تصل إلى **86.7%**، مما يشير إلى ضعف حاد في الجدارة الائتمانية للعميل وفجوة كبيرة بين الالتزامات المالية الحالية والقدرة على السداد المستقبلي.

**2. العوامل المؤثرة في تقييم المخاطر:**
يعود هذا التصنيف السلبي بشكل أساسي إلى تكرار حالات التأخر في السداد لست مرات، مما يعكس ضعف الانضباط المالي لدى العميل. كما تبرز نسبة الديون المرتفعة التي بلغت 85% من إجمالي الدخل كعامل ضغط إضافي، مقترنة بمعدل استخدام مرتفع جداً للائتمان بنسبة 95%. هذه المؤشرات مجتمعة تؤكد أن العميل يعاني من "اختناق ائتماني"، حيث يتم استهلاك معظم دخله وموارده الائتمانية المتاحة لتغطية التزامات قائمة، مما يرفع من حساسية وضعه المالي تجاه أي هزات نقدية بسيطة.

**

In [29]:

#  Test: Low Risk Client
low_risk_client = {
    'util_rate'            : 0.15,
    'age'                  : 45,
    'late_30_59'           : 0,
    'debt_ratio'           : 0.2,
    'monthly_income'       : 8000,
    'open_credits'         : 4,
    'late_90'              : 0,
    'real_estate_loans'    : 1,
    'late_60_89'           : 0,
    'dependents'           : 1,
    'debt_to_income'       : 0.2 / (8000 + 1),
    'total_late'           : 0,
    'has_late_history'     : 0,
    'income_per_dependent' : 8000,
    'high_util'            : 0
}

result_low = explain_client(low_risk_client, language='en')

print(f"Risk Score : {result_low['risk_score']:.1%}")
print(f"Risk Label : {result_low['risk_label']}")
print(f"\n{'='*50}")
print("LLM REPORT:")
print('='*50)
print(result_low['llm_report'])

Risk Score : 5.0%
Risk Label : Low Risk

LLM REPORT:
**Credit Risk Assessment Report**

**Executive Summary and Final Classification**
Following a comprehensive review of the applicant’s profile and the results from our predictive risk modeling, the client has been classified as **Low Risk**. With an estimated default probability of 5.0%, the applicant demonstrates a high degree of financial stability and creditworthiness. The client’s profile—characterized by a mature age (45), a robust monthly income of $8,000, and a flawless payment record—aligns with our institution's standards for high-quality borrowers. The model’s low default projection reflects a strong likelihood of consistent repayment throughout the loan lifecycle.

**Analysis of Key Risk Drivers**
The primary drivers behind this favorable risk assessment are the client’s exceptional credit discipline and conservative leverage. Specifically, the **zero late payment count** serves as the strongest indicator of reliability, si

In [30]:

# Save llm_explainer.py
llm_explainer_code = '''import pickle
import json
import numpy as np
import pandas as pd
import google.generativeai as genai

with open("model/final_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("model/shap_explainer.pkl", "rb") as f:
    explainer = pickle.load(f)
with open("model/feature_cols.json", "r") as f:
    feature_cols = json.load(f)


def shap_to_factors(shap_vals, feature_names, top_n=5):
    labels = {
        "util_rate"            : ("معدل استخدام الائتمان", "credit utilization rate"),
        "total_late"           : ("عدد مرات التأخر",        "late payment count"),
        "has_late_history"     : ("وجود تاريخ تأخر",        "late payment history"),
        "age"                  : ("العمر",                  "age"),
        "debt_ratio"           : ("نسبة الديون",            "debt ratio"),
        "monthly_income"       : ("الدخل الشهري",           "monthly income"),
        "debt_to_income"       : ("نسبة الدين للدخل",       "debt-to-income ratio"),
        "open_credits"         : ("عدد خطوط الائتمان",      "open credit lines"),
        "late_90"              : ("تأخر أكثر من 90 يوم",    "90+ days late"),
        "late_30_59"           : ("تأخر 30-59 يوم",         "30-59 days late"),
        "late_60_89"           : ("تأخر 60-89 يوم",         "60-89 days late"),
        "income_per_dependent" : ("الدخل لكل معال",         "income per dependent"),
        "high_util"            : ("استخدام ائتمان مرتفع",   "high credit usage"),
        "real_estate_loans"    : ("قروض عقارية",            "real estate loans"),
        "dependents"           : ("عدد المعالين",           "number of dependents"),
    }
    pairs = sorted(zip(shap_vals, feature_names),
                   key=lambda x: abs(x[0]), reverse=True)[:top_n]
    factors = []
    for sv, fn in pairs:
        ar, en = labels.get(fn, (fn, fn))
        factors.append({
            "feature"   : fn,
            "ar_label"  : ar,
            "en_label"  : en,
            "direction" : "increases" if sv > 0 else "decreases",
            "impact"    : round(abs(sv), 4),
            "shap_val"  : round(sv, 4)
        })
    return factors


def explain_client(client_dict: dict, language: str = "en",
                   api_key: str = "") -> dict:

    client_df  = pd.DataFrame([client_dict])[feature_cols]
    risk_score = float(model.predict_proba(client_df)[:, 1][0])
    shap_exp   = explainer(client_df)
    factors    = shap_to_factors(shap_exp.values[0], feature_cols)

    risk_label = (
        "High Risk"   if risk_score > 0.6 else
        "Medium Risk" if risk_score > 0.3 else
        "Low Risk"
    )
    risk_label_ar = (
        "مخاطرة عالية"   if risk_score > 0.6 else
        "مخاطرة متوسطة"  if risk_score > 0.3 else
        "مخاطرة منخفضة"
    )

    factors_text_en = "\\n".join([
        f"- {f[chr(39)]en_label{chr(39)]}: {f[chr(39)]direction{chr(39)]}} default risk (impact: {f[chr(39)]impact{chr(39)]}})"
        for f in factors
    ])
    factors_text_ar = "\\n".join([
        f"- {f[chr(39)]ar_label{chr(39)]}: {chr(39)]يزيد{chr(39)]} if f[chr(39)]direction{chr(39)]}==chr(39)]increases{chr(39)]} else {chr(39)]يقلل{chr(39)]}  (تأثير: {f[chr(39)]impact{chr(39)]}})"
        for f in factors
    ])

    if language == "ar":
        prompt = f"""أنت محلل مخاطر ائتمانية محترف. اكتب تقريراً من 3 فقرات:
درجة المخاطرة: {risk_score:.1%} — {risk_label_ar}
أهم العوامل:
{factors_text_ar}
1. ملخص وضع العميل  2. أهم العوامل  3. توصية للبنك (موافقة/رفض/بشروط)
اكتب بالعربية الفصحى."""
    else:
        prompt = f"""You are a credit risk analyst. Write a 3-paragraph report:
Default Probability: {risk_score:.1%} — {risk_label}
Key Factors:
{factors_text_en}
1. Executive summary  2. Key drivers  3. Recommendation (Approve/Reject/Conditional)"""

    genai.configure(api_key=api_key)
    gemini_model = genai.GenerativeModel("gemini-2.0-flash")
    response     = gemini_model.generate_content(prompt)

    return {
        "risk_score" : round(risk_score, 4),
        "risk_label" : risk_label,
        "factors"    : factors,
        "llm_report" : response.text,
        "language"   : language
    }
'''

with open('llm_explainer.py', 'w', encoding='utf-8') as f:
    f.write(llm_explainer_code)

print("✓ llm_explainer.py saved")
files.download('llm_explainer.py')

✓ llm_explainer.py saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>